# HR Onboarding AI Agent

This notebook documents the development of an HR onboarding assistant using an OpenAI language model, LangGraph, a RAG knowledge base, and an MCP employee directory. The final sections examine tool selection and document retrieval.

## Environment and dependencies

The following libraries provide the components used throughout the application. Package installation is kept in a single cell so that the notebook remains reproducible without repeating installations.

In [1]:
%pip install -qU \
    langchain \
    langgraph \
    langchain-openai \
    langchain-community \
    langchain-chroma \
    langchain-text-splitters \
    chromadb \
    pypdf \
    langsmith \
    langchain-mcp-adapters \
    mcp \
    pandas \
    python-dotenv

print("Dependencies installed successfully.")

Note: you may need to restart the kernel to use updated packages.
Dependencies installed successfully.


  You can safely remove it manually.


## Application configuration

Environment variables were used to keep API credentials outside the notebook. The `.env` file holds the required configuration values, while the code loads them into the Python environment.

In [2]:
import os
from dotenv import load_dotenv

# Project-level environment file containing OPENAI_API_KEY and LANGSMITH_API_KEY.
ENV_PATH = r"C:\Users\DELL\OneDrive\Documents\thesis\.env"

if not os.path.exists(ENV_PATH):
    raise FileNotFoundError(f"Environment file not found: {ENV_PATH}")

load_dotenv(ENV_PATH)

# LangSmith configuration for observability.
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGSMITH_PROJECT"] = "Thesis HR agent"

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("OPENAI_API_KEY is missing from the .env file.")

print("OpenAI API key available:", bool(os.getenv("OPENAI_API_KEY")))
print("LangSmith API key available:", bool(os.getenv("LANGSMITH_API_KEY")))
print("LangSmith project:", os.environ["LANGSMITH_PROJECT"])

OpenAI API key available: True
LangSmith API key available: True
LangSmith project: Thesis HR agent


## Project resources

The HR Policy Manual was used as the document-based knowledge source. A separate employee dataset was used to simulate structured directory information. Paths to these files and the local databases were defined in one place.

In [3]:
PROJECT_DIR = r"C:\Users\DELL\OneDrive\Documents\thesis"

HR_PDF_PATH = os.path.join(PROJECT_DIR, "HR Policy Manual 2023 (8).pdf")
EMPLOYEE_CSV_PATH = os.path.join(PROJECT_DIR, "employee_records.csv")
SQLITE_PATH = os.path.join(PROJECT_DIR, "employee_directory.db")
VECTOR_DB_PATH = os.path.join(PROJECT_DIR, "hr_vector_db_eval")

required_files = {
    "HR Policy Manual": HR_PDF_PATH,
    "Employee dataset": EMPLOYEE_CSV_PATH,
}

for name, path in required_files.items():
    print(f"{name}: {os.path.exists(path)}")

missing = [name for name, path in required_files.items() if not os.path.exists(path)]
if missing:
    raise FileNotFoundError(f"Missing project resources: {', '.join(missing)}")

HR Policy Manual: True
Employee dataset: True


## HR Policy Manual knowledge base

The HR manual was loaded from PDF and split into overlapping text passages. Each passage was given a unique `chunk_id` to support retrieval evaluation. The passages were converted into embeddings and stored in Chroma so the agent could find relevant HR information.

In [4]:
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma
import shutil

# Load the PDF and preserve page metadata for traceability.
loader = PyPDFLoader(HR_PDF_PATH)
documents = loader.load()

# Overlapping chunks help preserve context across chunk boundaries.
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=150,
)
chunks = text_splitter.split_documents(documents)
# Assign a unique ID to each HR Manual chunk

for i, chunk in enumerate(chunks):
    chunk.metadata["chunk_id"] = f"hr_chunk_{i:04d}"

print(f"Total chunks: {len(chunks)}")
print("First chunk ID:", chunks[0].metadata["chunk_id"])

print(f"PDF pages loaded: {len(documents)}")
print(f"Text chunks created: {len(chunks)}")

C:\Users\DELL\AppData\Local\Temp\ipykernel_13264\1381023647.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


Total chunks: 550
First chunk ID: hr_chunk_0000
PDF pages loaded: 208
Text chunks created: 550


In [5]:
# Set this to True only when the vector database needs to be rebuilt.
RESET_VECTOR_DB = False
VECTOR_DB_PATH = os.path.join(
    PROJECT_DIR,
    "hr_vector_db_evaluation_new"
)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

if RESET_VECTOR_DB and os.path.exists(VECTOR_DB_PATH):
    shutil.rmtree(VECTOR_DB_PATH)

if os.path.exists(VECTOR_DB_PATH):
    vector_db = Chroma(
        collection_name="hr_manual",
        embedding_function=embeddings,
        persist_directory=VECTOR_DB_PATH,
    )
    print("Existing Chroma vector database loaded.")
else:
    vector_db = Chroma.from_documents(
        documents=chunks,
        embedding=embeddings,
        persist_directory=VECTOR_DB_PATH,
        collection_name="hr_manual",
    )
    print("New Chroma vector database created.")

Existing Chroma vector database loaded.


In [6]:
docs = vector_db.similarity_search(
    "annual leave policy",
    k=4
)

for i, doc in enumerate(docs, start=1):
    print(f"Result {i}")
    print("Chunk ID:", doc.metadata.get("chunk_id"))
    print("Page:", doc.metadata.get("page"))
    print("-------------------")

Result 1
Chunk ID: hr_chunk_0016
Page: 6
-------------------
Result 2
Chunk ID: hr_chunk_0223
Page: 78
-------------------
Result 3
Chunk ID: hr_chunk_0226
Page: 79
-------------------
Result 4
Chunk ID: hr_chunk_0239
Page: 83
-------------------


In [7]:
for i, doc in enumerate(docs, start=1):
    print(f"\n{'=' * 60}")
    print(f"RESULT {i}")
    print("Chunk ID:", doc.metadata.get("chunk_id"))
    print("Page:", doc.metadata.get("page"))
    print("-" * 60)
    print(doc.page_content)


RESULT 1
Chunk ID: hr_chunk_0016
Page: 6
------------------------------------------------------------
v
IIMA HR Policy Manual 2023
Chapter 6 Leave and Attendance ..............................................................68
(1) Office Hours  .....................................................................................................................................68
(2) Procedure for granting leave .....................................................................................................68
(3) Extension of Leave  ........................................................................................................................69
(4) Cancellation of Leave ....................................................................................................................69
(5) Kinds of Leave ..................................................................................................................................69
5.1 Leave Type 1: Casual Leave ......

In [8]:
# Find chunks mentioning Earned Leave

for chunk in chunks:
    content = chunk.page_content.lower()

    if "earned leave" in content:
        print("=" * 60)
        print("Chunk ID:", chunk.metadata["chunk_id"])
        print("Page:", chunk.metadata.get("page"))
        print("Content:", chunk.page_content[:600])

Chunk ID: hr_chunk_0017
Page: 6
Content: 5.1 Leave Type 1: Casual Leave ........................................................................................................69
5.2 Leave Type 2: Earned Leave  ......................................................................................................70
5.3 Leave Type 3: Half Pay Leave .....................................................................................................71
5.4 Leave Type 4: Commuted Leave  ..............................................................................................71
5.5 Leave Type 5: Extraordinary Leave ...................
Chunk ID: hr_chunk_0018
Page: 6
Content: 5.7 Leave Type 7: Paternity Leave ...................................................................................................73
5.8  Leave Type 8: Leave to female employees on adoption of child ................................74
(6) Encashment of Earned Leave .................................................

## HR Manual retrieval tool

A retrieval function was exposed as the `search_hr_manual` tool. It searches the Chroma knowledge base and returns relevant HR passages when the agent needs information from the manual.

In [9]:
from langchain.tools import tool

@tool
def search_hr_manual(query: str) -> str:
    """Search the company HR Policy Manual for relevant information.

    Use this tool for company-specific HR policies, leave, benefits,
    working conditions, onboarding procedures, compensation,
    travel policies, and related HR rules.
    """

    docs = vector_db.similarity_search(query, k=4)

    if not docs:
        return "No relevant information was found in the HR Policy Manual."

    results = []
    for index, doc in enumerate(docs, start=1):
        page = doc.metadata.get("page")
        page = page + 1 if isinstance(page, int) else "Unknown"

        results.append(
            f"HR Manual result {index}\n"
            f"Page: {page}\n"
            f"{doc.page_content}"
        )

    return "\n\n".join(results)

print("HR Manual tool available as:", search_hr_manual.name)

HR Manual tool available as: search_hr_manual


## Employee Directory data layer

Employee records were prepared as structured data in a local SQLite database. This made it possible to retrieve specific employee information separately from the HR manual.

In [10]:
import pandas as pd
import sqlite3

employees_df = pd.read_csv(EMPLOYEE_CSV_PATH)

print(f"Employee records: {len(employees_df):,}")
print("Available fields:")
print(employees_df.columns.tolist())

# Recreate the local SQLite table from the source dataset.
with sqlite3.connect(SQLITE_PATH) as conn:
    employees_df.to_sql(
        "employees",
        conn,
        if_exists="replace",
        index=False,
    )

print("SQLite Employee Directory database ready.")

Employee records: 30,000
Available fields:
['Employee_ID', 'Employee_Name', 'Age', 'Country', 'Department', 'Position', 'Salary', 'Joining_Date']
SQLite Employee Directory database ready.


## Employee Directory MCP server

A separate Python script, `employee_directory_server.py`, was used to expose employee-directory functions through MCP. The server runs outside the notebook and provides the structured-data tools used by the agent.

## MCP connection and tool discovery

The notebook connected to the local MCP server and loaded its available employee-directory tools. These tools were then made available to the agent alongside the HR retrieval tool.

In [11]:
from langchain_mcp_adapters.client import MultiServerMCPClient

mcp_client = MultiServerMCPClient(
    {
        "employee_directory": {
            "transport": "streamable_http",
            "url": "http://localhost:8000/mcp",
        }
    }
)

directory_tools = await mcp_client.get_tools()

print("Employee Directory MCP tools:")
for tool in directory_tools:
    print(f"- {tool.name}")

Employee Directory MCP tools:
- search_employee
- find_employees_by_department
- get_employee_profile


## OpenAI model and agent configuration

The OpenAI model was used to understand questions and generate responses. LangGraph coordinated the agent and its available tools, allowing different information sources to be used when needed.

In [12]:
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent

llm = ChatOpenAI(
    model="gpt-5-mini",
    temperature=0.2,
)

# The agent can use both the document-retrieval capability and MCP capabilities.
all_tools = [search_hr_manual, *directory_tools]

print("Tools available to the agent:")
for available_tool in all_tools:
    print(f"- {available_tool.name}")

Tools available to the agent:
- search_hr_manual
- search_employee
- find_employees_by_department
- get_employee_profile


## Agent behavior

System instructions were defined to guide the agent in selecting tools. HR policy questions were directed to the manual, employee-specific questions to the directory, and general questions could be answered without organizational tools.

In [13]:
SYSTEM_PROMPT = """
You are an AI HR Onboarding Agent designed to assist employees during their onboarding journey.

You have access to two types of company information:

1. HR Manual RAG tool:
   Use search_hr_manual for company-specific HR policies, leave,
   benefits, working conditions, onboarding procedures,
   compensation, travel policies, and HR rules.

2. Employee Directory MCP tools:
   Use these tools for employee names, profiles, departments,
   positions, and directory information.

Guidelines:
- Do not invent company-specific information.
- Use the HR Manual when the answer depends on HR policy.
- Use Employee Directory tools when employee information is required.
- If no external information is required, answer directly.
- Use tools whenever they provide information necessary for a reliable answer.
- Be professional, clear, concise, and employee-friendly.
"""

memory = InMemorySaver()

hr_agent = create_agent(
    model=llm,
    tools=all_tools,
    system_prompt=SYSTEM_PROMPT,
    checkpointer=memory,
)

print("HR onboarding agent initialized successfully.")

HR onboarding agent initialized successfully.


## Conversational interface

An asynchronous interaction function was created to send messages to the agent and receive responses. A conversation identifier (`thread_id`) was used to maintain short-term context across related questions.

In [14]:
async def ask_hr_agent(question: str, thread_id: str = "employee_001"):
    """Send an employee request to the HR onboarding agent."""

    config = {
        "configurable": {
            "thread_id": thread_id,
        }
    }

    response = await hr_agent.ainvoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": question,
                }
            ]
        },
        config=config,
    )

    answer = response["messages"][-1].content
    print(answer)
    return response

In [15]:
print("Available agent tools:")

for tool in all_tools:
    print("-", tool.name)

Available agent tools:
- search_hr_manual
- search_employee
- find_employees_by_department
- get_employee_profile


## Evaluation 1 — Tool selection

A small synthetic dataset of five questions was prepared to test whether the agent selected the appropriate tools. The expected categories were **RAG** for HR policies, **MCP** for employee records, **BOTH** for combined questions, and **NONE** for general questions. Each question was evaluated in a separate conversation to avoid interference from previous messages. The actual tool calls were compared with the expected categories to calculate **Tool Call Accuracy**. This metric measures tool choice, not the accuracy of the final answer. The MCP server needs to be running for the experiment to execute.

In [16]:
import pandas as pd

evaluation_data = [
    {
        "question_id": "Q001",
        "question": "What is the annual leave policy?",
        "expected_tool_category": "RAG"
    },
    {
        "question_id": "Q002",
        "question": "What is the probation period according to the HR Manual?",
        "expected_tool_category": "RAG"
    },
    {
        "question_id": "Q003",
        "question": "Which country is Ethan Davis, Employee ID 19, based in?",
        "expected_tool_category": "MCP"
    },
    {
        "question_id": "Q004",
        "question": "What is the capital of France?",
        "expected_tool_category": "NONE"
    },
    {
        "question_id": "Q005",
        "question": (
            "What is the annual leave policy, and which country "
            "is employee 19 based in?"
        ),
        "expected_tool_category": "BOTH"
    }
]

df_eval = pd.DataFrame(evaluation_data)

display(df_eval)

,question_id,question,expected_tool_category
0,Q001,What is the annual leave policy?,RAG
1,Q002,What is the probation period according to the ...,RAG
2,Q003,"Which country is Ethan Davis, Employee ID 19, ...",MCP
3,Q004,What is the capital of France?,NONE
4,Q005,"What is the annual leave policy, and which cou...",BOTH


In [17]:
def extract_tool_calls(response):

    called_tools = []

    for message in response["messages"]:

        if getattr(message, "type", None) == "ai":

            for call in getattr(message, "tool_calls", []):
                called_tools.append(call["name"])

    return called_tools

In [18]:
def classify_tool_calls(called_tools):

    rag_used = "search_hr_manual" in called_tools

    mcp_tool_names = {
        tool.name for tool in directory_tools
    }

    mcp_used = any(
        name in mcp_tool_names
        for name in called_tools
    )

    if rag_used and mcp_used:
        return "BOTH"

    if rag_used:
        return "RAG"

    if mcp_used:
        return "MCP"

    return "NONE"

In [19]:
tool_evaluation_results = []

for _, row in df_eval.iterrows():

    question_id = row["question_id"]
    question = row["question"]

    config = {
        "configurable": {
            "thread_id": f"eval_{question_id}"
        }
    }

    response = await hr_agent.ainvoke(
        {
            "messages": [
                {
                    "role": "user",
                    "content": question
                }
            ]
        },
        config=config
    )

    called_tools = extract_tool_calls(response)

    actual_category = classify_tool_calls(called_tools)

    expected_category = row["expected_tool_category"]

    correct = actual_category == expected_category

    tool_evaluation_results.append({
        "question_id": question_id,
        "question": question,
        "expected_category": expected_category,
        "actual_category": actual_category,
        "called_tools": called_tools,
        "correct": correct
    })

    print(f"{question_id} completed")

Q001 completed
Q002 completed
Q003 completed
Q004 completed
Q005 completed


In [20]:
df_tool_results = pd.DataFrame(tool_evaluation_results)

display(df_tool_results)

accuracy = df_tool_results["correct"].mean()

print(f"Tool Call Accuracy: {accuracy:.2%}")

,question_id,question,expected_category,actual_category,called_tools,correct
0,Q001,What is the annual leave policy?,RAG,RAG,"[search_hr_manual, search_hr_manual, search_hr...",True
1,Q002,What is the probation period according to the ...,RAG,RAG,"[search_hr_manual, search_hr_manual]",True
2,Q003,"Which country is Ethan Davis, Employee ID 19, ...",MCP,MCP,[get_employee_profile],True
3,Q004,What is the capital of France?,NONE,NONE,[],True
4,Q005,"What is the annual leave policy, and which cou...",BOTH,BOTH,"[search_hr_manual, get_employee_profile, searc...",True


Tool Call Accuracy: 100.00%


In [21]:
df_tool_results.to_csv(
    "tool_selection_results.csv",
    index=False
)

## Evaluation 2 — Manual RAG retrieval assessment

Five HR policy questions were selected from the prepared evaluation dataset. Chroma retrieved the **four highest-ranked passages** for each question, giving **20 passages** in total. Each passage was assessed manually against the question and reference answer. A passage was labeled **1** when it provided factual evidence for the answer (including a necessary part of it), or **0** when it did not. No LLM was used to judge passage relevance.

The manual labels were used to calculate **Precision@4** and **Hit Rate@4**. These results describe only the five selected questions, not the entire HR manual. Recall@4 was not calculated because all relevant passages in the manual had not been identified.

### Selected questions and retrieved passages

The existing evaluation CSV provided the questions and reference answers. Five question IDs were selected (`R001`, `R004`, `R005`, `R010`, `R016`), and Chroma returned four full passages for each question. The passage text and its unique ID were stored to support the manual review.

In [22]:
import os
import pandas as pd

EVAL_DATASET_PATH = os.path.join(PROJECT_DIR, "hr_rag_evaluation_20_questions.csv")
rag_eval_df = pd.read_csv(EVAL_DATASET_PATH)
required = {"question_id", "question", "reference_answer"}
if not required.issubset(rag_eval_df.columns):
    raise ValueError(f"Evaluation CSV must contain: {sorted(required)}")

selected_questions = ["R001", "R004", "R005", "R010", "R016"]
selected_df = rag_eval_df[rag_eval_df["question_id"].isin(selected_questions)].copy()
if len(selected_df) != 5 or not selected_df["question_id"].is_unique:
    raise ValueError("The five selected question IDs must each appear once in the CSV.")

rows = []
for _, item in selected_df.iterrows():
    passages = vector_db.similarity_search(str(item["question"]), k=4)
    if len(passages) != 4:
        raise ValueError(f"Expected four passages for {item['question_id']}.")
    for rank, passage in enumerate(passages, start=1):
        rows.append({
            "question_id": item["question_id"],
            "question": item["question"],
            "reference_answer": item["reference_answer"],
            "rank": rank,
            "retrieved_chunk_id": passage.metadata.get("chunk_id"),
            "page_index": passage.metadata.get("page"),
            "content": passage.page_content,
        })

manual_sample = pd.DataFrame(rows).sort_values(["question_id", "rank"]).reset_index(drop=True)
if manual_sample["retrieved_chunk_id"].isna().any():
    raise ValueError("Missing chunk IDs. Rebuild the Chroma database with chunk_id metadata.")
print(f"Selected questions: {len(selected_df)} | Retrieved passages: {len(manual_sample)}")
display(manual_sample[["question_id", "rank", "retrieved_chunk_id", "page_index"]])

Selected questions: 5 | Retrieved passages: 20


,question_id,rank,retrieved_chunk_id,page_index
0,R001,1,hr_chunk_0227,79
1,R001,2,hr_chunk_0228,79
2,R001,3,hr_chunk_0017,6
3,R001,4,hr_chunk_0236,82
4,R004,1,hr_chunk_0226,79
5,R004,2,hr_chunk_0227,79
6,R004,3,hr_chunk_0225,78
7,R004,4,hr_chunk_0017,6
8,R005,1,hr_chunk_0227,79
9,R005,2,hr_chunk_0226,79


### Manual relevance labels

A CSV table was created containing the 20 retrieved passages and an empty `manual_relevant` column. The researcher reviewed the full text of each passage and recorded **1** for relevant or **0** for not relevant. The CSV was kept between runs so the manual judgments were not overwritten.

The only field completed manually was `manual_relevant`; the question, reference answer, and retrieved passage were already included in the table.

In [24]:
manual_path = os.path.join(PROJECT_DIR, "rag_manual_review_20_passages.csv")
key_cols = ["question_id", "rank", "retrieved_chunk_id"]

if os.path.exists(manual_path):
    print("Existing manual review CSV preserved:", manual_path)
else:
    export_df = manual_sample.copy()
    export_df["manual_relevant"] = pd.NA
    export_df.to_csv(manual_path, index=False, encoding="utf-8-sig")
    print("Manual review CSV created:", manual_path)

display(manual_sample[["question_id", "question", "rank", "retrieved_chunk_id"]])

Existing manual review CSV preserved: C:\Users\DELL\OneDrive\Documents\thesis\rag_manual_review_20_passages.csv


,question_id,question,rank,retrieved_chunk_id
0,R001,How many days of Earned Leave are administrati...,1,hr_chunk_0227
1,R001,How many days of Earned Leave are administrati...,2,hr_chunk_0228
2,R001,How many days of Earned Leave are administrati...,3,hr_chunk_0017
3,R001,How many days of Earned Leave are administrati...,4,hr_chunk_0236
4,R004,How many days of Casual Leave can an employee ...,1,hr_chunk_0226
5,R004,How many days of Casual Leave can an employee ...,2,hr_chunk_0227
6,R004,How many days of Casual Leave can an employee ...,3,hr_chunk_0225
7,R004,How many days of Casual Leave can an employee ...,4,hr_chunk_0017
8,R005,Can unused Casual Leave be carried forward to ...,1,hr_chunk_0227
9,R005,Can unused Casual Leave be carried forward to ...,2,hr_chunk_0226


### Manual retrieval metrics

After the 20 relevance judgments were recorded, the completed CSV was loaded and checked. **Precision@4** measured how many of the four retrieved passages were relevant for each question. **Hit Rate@4** measured whether at least one relevant passage was retrieved. The final values were averaged across the five questions.

In [25]:
manual_labeled = pd.read_csv(manual_path)
if len(manual_labeled) != 20 or manual_labeled.duplicated(key_cols).any():
    raise ValueError("The manual review file must contain 20 unique passages.")

expected_keys = set(map(tuple, manual_sample[key_cols].itertuples(index=False, name=None)))
actual_keys = set(map(tuple, manual_labeled[key_cols].itertuples(index=False, name=None)))
if expected_keys != actual_keys:
    raise ValueError("The saved labels do not match the current retrieved passages.")

labels = pd.to_numeric(manual_labeled["manual_relevant"], errors="coerce")
if labels.isna().any() or not labels.isin([0, 1]).all():
    raise ValueError("Complete manual_relevant with 0 or 1 for all 20 passages.")
manual_labeled["manual_relevant"] = labels.astype(int)

rag_metrics_df = (manual_labeled.groupby("question_id")
    .agg(relevant_retrieved=("manual_relevant", "sum"),
         retrieved_count=("manual_relevant", "size"))
    .reset_index())
if not (rag_metrics_df["retrieved_count"] == 4).all():
    raise ValueError("Each question must have exactly four reviewed passages.")

rag_metrics_df["precision_at_4"] = rag_metrics_df["relevant_retrieved"] / 4
rag_metrics_df["hit_rate_at_4"] = (rag_metrics_df["relevant_retrieved"] > 0).astype(int)
display(rag_metrics_df)
print(f"Manual Precision@4: {rag_metrics_df['precision_at_4'].mean():.2%}")
print(f"Manual Hit Rate@4: {rag_metrics_df['hit_rate_at_4'].mean():.2%}")
rag_metrics_df.to_csv(os.path.join(PROJECT_DIR, "rag_manual_sample_metrics.csv"), index=False)

,question_id,relevant_retrieved,retrieved_count,precision_at_4,hit_rate_at_4
0,R001,1,4,0.25,1
1,R004,1,4,0.25,1
2,R005,1,4,0.25,1
3,R010,1,4,0.25,1
4,R016,2,4,0.50,1


Manual Precision@4: 30.00%
Manual Hit Rate@4: 100.00%


### Evaluation summary and limitations

Two parts of the prototype were evaluated. **Tool Call Accuracy** compared the tools chosen by the agent with expected categories for five synthetic questions. **Manual RAG evaluation** measured the relevance of 20 retrieved passages from five selected HR policy questions. The two evaluations assessed different aspects of the system: tool selection and document retrieval.

The datasets were small and synthetic, and the RAG sample was selected deliberately rather than randomly. The results therefore provide an initial feasibility assessment, not proof of general performance. Manual relevance labels were used only for retrieval metrics; they did not measure the correctness of the agent's final written answers.

## Agent interaction examples

Example questions were included to demonstrate how the agent handled HR policy retrieval, employee-directory queries, and questions that did not require organizational tools.

In [ ]:
await ask_hr_agent(
    "What is the annual leave policy?",
    thread_id="employee_001",
)

In [ ]:
await ask_hr_agent(
    "Find employees working in HR.",
    thread_id="employee_001",
)

In [ ]:
await ask_hr_agent(
    "What are the eligibility requirements for maternity leave according to the HR policy",
    thread_id="employee_001",
)

In [ ]:
await ask_hr_agent(
    "According to the HR Manual, what should a new employee do if they have a question about an HR policy or procedure?",
    thread_id="employee_001",
)

In [ ]:
await ask_hr_agent(
    "According to the HR Manual, what is the company's revenue for 2025??",
    thread_id="employee_001",
)

## Conversational memory demonstration

Messages sharing the same `thread_id` were used to illustrate short-term conversational memory. This allowed follow-up questions to use information from the preceding interaction.

In [ ]:
await ask_hr_agent(
    "What information can you give me about employees in that department?",
    thread_id="employee_001",
)

In [ ]:
await ask_hr_agent(
    "What department does Ethan Davis work in?",
    thread_id="employee_001",
)

In [ ]:
await ask_hr_agent(
    "Which country is Ethan Davis (Employee ID 19) based in?",
    thread_id="employee_001",
)

In [ ]:
await ask_hr_agent(
    "Can you give me Ethan Davis's department, position, and country??",
    thread_id="employee_001",
)

In [ ]:
await ask_hr_agent(
    "Can you give me the employee information for John Anderson?",
    thread_id="employee_001",
)

In [ ]:
await ask_hr_agent(
    "What does the HR Manual say about probation periods?",
    thread_id="employee_001",
)

In [ ]:
await ask_hr_agent(
    "What is the difference between an employee and an employer",
    thread_id="employee_001",
)

In [ ]:
await ask_hr_agent(
    "What does the HR Manual say about probation periods, and what department does Ethan Davis (Employee ID 19) work in?",
    thread_id="employee_001",
)

In [ ]:
await ask_hr_agent(
    "What will my salary be after one year in the company?",
    thread_id="employee_001",
)

## System architecture

The completed prototype combined a language model, LangGraph orchestration, Chroma retrieval, an MCP employee directory, and conversational memory. Each component had a distinct role in providing access to HR onboarding information.

## Conclusion

The prototype brought document retrieval and structured employee-data access into one conversational assistant. The added evaluation sections examined whether it selected appropriate tools and whether retrieved HR passages supported the reference answers. The reported metrics describe the tested synthetic scenarios and do not establish real-world deployment performance.